# 4D Chebyshev interpolation

Four dimensions with a modest grid is already ~20k samples, but the
construction is still a single DCT. Pruning and precomputation become
increasingly valuable here: a full 4D Clenshaw evaluation touches every kept
coefficient, so shrinking the block and collapsing fixed axes pay off
directly.

As in lower dimensions, axis *k* of the value array is variable *k* and the
bounds are passed per axis in the same order. Every axis has a different
number of nodes, so an axis mix-up cannot pass unnoticed.

In [1]:
import numpy as np
from time import perf_counter

from chebyshevnd import ChebyshevInterpolant4D, chebyshev_nodes

In [2]:
f = lambda x, y, z, w: np.sin(x) * np.cos(y) * np.exp(-z) * (w**3 + w)
domains = [(0.0, 2.0), (-1.0, 3.0), (1.0, 4.0), (-2.0, 0.0)]   # (lower, upper) for x, y, z, w
Ns = (13, 14, 12, 8)                                             # nodes per axis, same order

axes = [chebyshev_nodes(n, a, b) for n, (a, b) in zip(Ns, domains)]
grids = np.meshgrid(*axes, indexing="ij")
vals = f(*grids)
assert vals.shape == Ns   # axis k of vals is variable k

(ax_, bx_), (ay_, by_), (az_, bz_), (aw_, bw_) = domains
interp = ChebyshevInterpolant4D(vals, ax=ax_, bx=bx_, ay=ay_, by=by_,
                                az=az_, bz=bz_, aw=aw_, bw=bw_)
print(f"grid points sampled: {np.prod(Ns)}")

grid points sampled: 17472


In [3]:
rng = np.random.default_rng(1)
points = [tuple(rng.uniform(a, b) for a, b in domains) for _ in range(100)]
err = max(abs(interp.evaluate(*p) - f(*p)) for p in points)
print(f"max error at 100 random points: {err:.3e}")
print(f"estimated absolute error:       {interp.absolute_error():.3e}")

max error at 100 random points: 9.279e-11
estimated absolute error:       8.755e-08


## Pruning

In [4]:
print(f"full grid:      {Ns}")
interp.optimize(rel_tol=1e-8)
print(f"effective grid: {interp.effective_grid()}")

err_opt = max(abs(interp.evaluate(*p) - f(*p)) for p in points)
print(f"max error after pruning: {err_opt:.3e}")

full grid:      (13, 14, 12, 8)
effective grid: (10, 13, 12, 4)
max error after pruning: 7.745e-10


## Precomputing a dimension

`precompute_x` collapses the leading axis at a fixed value; `evaluate_yzw`
then only does 3D work per call.

In [5]:
x0 = 0.7
interp.precompute_x(x0)

y0, z0, w0 = 1.0, 2.0, -1.0
print(f"evaluate:     {interp.evaluate(x0, y0, z0, w0):.12f}")
print(f"evaluate_yzw: {interp.evaluate_yzw(y0, z0, w0):.12f}")

evaluate:     -0.094212927195
evaluate_yzw: -0.094212927195


In [6]:
yzw_points = [tuple(rng.uniform(a, b) for a, b in domains[1:]) for _ in range(1000)]

t0 = perf_counter()
for y, z, w in yzw_points:
    interp.evaluate(x0, y, z, w)
t_full = perf_counter() - t0

t0 = perf_counter()
for y, z, w in yzw_points:
    interp.evaluate_yzw(y, z, w)
t_pre = perf_counter() - t0

print(f"1000 evaluations, full 4D:        {t_full*1e3:7.1f} ms")
print(f"1000 evaluations, precomputed x:  {t_pre*1e3:7.1f} ms")
print(f"speed-up: {t_full/t_pre:.1f}x")

1000 evaluations, full 4D:            5.9 ms
1000 evaluations, precomputed x:      1.1 ms
speed-up: 5.2x
